<a id="setup"></a>
# 1. Setup & Installation

Install required packages and load libraries.

In [1]:
# Install required packages
!pip install -q requests tqdm

# Import libraries
import os
import re
import json
import time
import logging
from typing import Optional, List, Dict, Any
from pathlib import Path

import requests
from tqdm.notebook import tqdm

print("✓ Libraries loaded successfully.")

✓ Libraries loaded successfully.


<a id="config"></a>
# 2. Configuration

Set up API configuration, model parameters, and file paths.

In [2]:
# API Configuration
AVALAI_API_KEY = os.environ.get("AVALAI_API_KEY", "aa-hXsC7ulscaBbPcU6663EvyHVCiyty0HKu2ar4UUXCVU0W89Y")
AVALAI_CHAT_URL = "https://api.avalai.ir/v1/chat/completions"
MODEL_NAME = "meta.llama3-1-70b-instruct-v1:0"

# Label set for fact-checking
CANON_LABELS = [
    "Supported",
    "Refuted",
    "Misleading",
    "Partially true"
]

# Retry configuration
MAX_RETRIES = 3
INITIAL_RETRY_DELAY = 0.5
SLEEP_BETWEEN_CALLS = 0.3  # Seconds between API calls

# Dataset paths
DATA_DIR = Path("../../../Datasets/AVeriTeC_FEVER")
INPUT_FILES = [
    DATA_DIR / "test_2023_2024.json",
    DATA_DIR / "test_2025.json"
]

# Output configuration
OUTPUT_DIR = Path("results_fever")
OUTPUT_DIR.mkdir(exist_ok=True)

print("✓ Configuration loaded successfully.")
print(f"Model: {MODEL_NAME}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Input files: {len(INPUT_FILES)}")

✓ Configuration loaded successfully.
Model: meta.llama3-1-70b-instruct-v1:0
Output directory: results_fever
Input files: 2


<a id="model-funcs"></a>
# 3. Model Functions

Define functions for label normalization and API interaction.

In [3]:
def normalize_label(label: Optional[str]) -> Optional[str]:
    """Normalize model predictions to canonical label set."""
    if label is None:
        return None
    s = str(label).strip().lower()
    
    # Direct mappings for model predictions (case-insensitive)
    direct = {
        "supported": "Supported",
        "refuted": "Refuted",
        "partially true": "Partially true",
        "misleading": "Misleading",
    }
    
    if s in direct:
        return direct[s]
    
    # Fallback with compact matching
    compact = re.sub(r"[^a-z]+", "", s)
    if compact == "supported":
        return "Supported"
    if compact == "refuted":
        return "Refuted"
    if compact in {"partiallytrue", "notenoughinformation", "insufficientevidence"}:
        return "Partially true"
    if compact.startswith("conflic") or "cherrypick" in compact or compact == "misleading":
        return "Misleading"
    
    return None


def ask_model(
    model: str,
    claim: str,
    claim_date: Optional[str] = None,
    logger: Optional[logging.Logger] = None,
    max_retries: int = 3,
    initial_delay: float = 0.5
) -> Optional[str]:
    """
    Query the model API with retry logic and exponential backoff.
    
    Args:
        model: Model name to use
        claim: Claim text to classify
        claim_date: Optional claim date for context
        logger: Optional logger for detailed logging
        max_retries: Maximum number of retry attempts
        initial_delay: Initial delay in seconds before first retry
    
    Returns:
        Normalized label or None if all retries fail
    """
    # Build prompt with optional date context
    date_context = f"\nClaim date: {claim_date}" if claim_date else ""
    
    user_prompt = (
        "You are a fact-checking classifier.\n"
        "Choose ONE label strictly from:\n"
        "Supported | Refuted | Misleading | Partially true\n\n"
        f"Claim:\n{claim}{date_context}\n\n"
        "Reply strictly as JSON: {\"label\": \"...\"} with no extra text."
    )
    
    if logger:
        logger.info(f"Asking model: {model}")
        logger.info(f"Claim: {claim[:200]}{'...' if len(claim) > 200 else ''}")
    
    for attempt in range(max_retries + 1):
        try:
            if attempt > 0:
                delay = initial_delay * (2 ** (attempt - 1))
                if logger:
                    logger.info(f"Retry attempt {attempt}/{max_retries} after {delay:.1f}s delay...")
                time.sleep(delay)
            
            resp = requests.post(
                AVALAI_CHAT_URL,
                headers={
                    "Authorization": f"Bearer {AVALAI_API_KEY}",
                    "Content-Type": "application/json",
                },
                json={
                    "model": model,
                    "messages": [
                        {"role": "system", "content": "You classify claims into the given label set."},
                        {"role": "user", "content": user_prompt},
                    ],
                    "temperature": 0.0,
                    "top_p": 0.0,
                    "max_tokens": 50,
                    "response_format": {"type": "json_object"},
                },
                timeout=45,
            )
            
            if logger:
                logger.info(f"API Response Status: {resp.status_code}")
            
            # Handle rate limiting (429)
            if resp.status_code == 429:
                if attempt < max_retries:
                    continue
                else:
                    if logger:
                        logger.error("Max retries reached for rate limit")
                    return None
            
            # Handle server errors (5xx)
            if 500 <= resp.status_code < 600:
                if attempt < max_retries:
                    continue
                else:
                    if logger:
                        logger.error(f"Max retries reached for server error {resp.status_code}")
                    return None
            
            # Handle other non-200 status codes
            if resp.status_code != 200:
                if logger:
                    logger.warning(f"API returned non-200 status: {resp.status_code}")
                return None
            
            # Parse successful response
            data = resp.json()
            txt = data.get("choices", [{}])[0].get("message", {}).get("content", "").strip()
            
            if logger:
                logger.info(f"Raw model response: {txt}")
            
            # Extract label from JSON response
            try:
                label_raw = json.loads(txt).get("label")
            except Exception as parse_err:
                if logger:
                    logger.warning(f"Failed to parse JSON response: {parse_err}")
                # Try regex extraction
                m = re.search(r'\"label\"\s*:\s*\"([^\"]+)\"', txt)
                if m:
                    label_raw = m.group(1)
                else:
                    if attempt < max_retries:
                        continue
                    return None
            
            # Normalize and return
            normalized = normalize_label(label_raw)
            if logger:
                logger.info(f"Extracted label: {label_raw} -> Normalized: {normalized}")
            
            if normalized is None and attempt < max_retries:
                if logger:
                    logger.warning("Label normalization failed, retrying...")
                continue
            
            return normalized
        
        except requests.exceptions.Timeout:
            if attempt < max_retries:
                continue
            else:
                if logger:
                    logger.error("Max retries reached for timeout")
                return None
        
        except requests.exceptions.ConnectionError as e:
            if attempt < max_retries:
                continue
            else:
                if logger:
                    logger.error("Max retries reached for connection error")
                return None
        
        except Exception as e:
            if logger:
                logger.error(f"Exception in ask_model (attempt {attempt + 1}): {e}")
            if attempt < max_retries:
                continue
            else:
                return None
    
    return None

print("✓ Model functions defined successfully.")

✓ Model functions defined successfully.


<a id="inference"></a>
# 4. Inference Pipeline

Define the main inference function to process FEVER dataset files.

In [4]:
def setup_logger(log_file: Path) -> logging.Logger:
    """Setup logger for inference run."""
    logger = logging.getLogger("fever_inference")
    logger.setLevel(logging.INFO)
    logger.handlers = []
    
    fh = logging.FileHandler(log_file, mode='w', encoding='utf-8')
    fh.setLevel(logging.INFO)
    formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
    fh.setFormatter(formatter)
    logger.addHandler(fh)
    
    return logger


def process_fever_file(
    input_file: Path,
    output_file: Path,
    model_name: str,
    logger: logging.Logger,
    sleep_between: float = 0.3
) -> Dict[str, Any]:
    """
    Process a FEVER dataset file and add model predictions.
    
    Args:
        input_file: Path to input JSON file
        output_file: Path to save output JSON file with predictions
        model_name: Model name to use for inference
        logger: Logger instance
        sleep_between: Sleep time between API calls (seconds)
    
    Returns:
        Dictionary with statistics about the inference run
    """
    logger.info(f"="*80)
    logger.info(f"Processing file: {input_file}")
    logger.info(f"Output file: {output_file}")
    logger.info(f"="*80)
    
    # Load input data
    logger.info("Loading input data...")
    with open(input_file, 'r', encoding='utf-8') as f:
        claims_data = json.load(f)
    
    total_claims = len(claims_data)
    logger.info(f"Loaded {total_claims} claims")
    
    # Process each claim
    results = []
    successful = 0
    failed = 0
    t0 = time.time()
    
    pbar = tqdm(claims_data, desc=f"Processing {input_file.name}", unit="claim")
    
    for idx, claim_entry in enumerate(pbar):
        logger.info(f"\n{'='*80}")
        logger.info(f"Processing claim #{idx} (ID: {claim_entry.get('claim_id', 'N/A')})")
        
        claim_text = claim_entry.get("claim", "")
        claim_date = claim_entry.get("claim_date")
        
        if not claim_text or not claim_text.strip():
            logger.warning(f"Empty claim at index {idx}, skipping")
            result_entry = claim_entry.copy()
            result_entry["prediction"] = None
            result_entry["prediction_error"] = "Empty claim"
            results.append(result_entry)
            failed += 1
            continue
        
        logger.info(f"Claim: {claim_text[:200]}{'...' if len(claim_text) > 200 else ''}")
        logger.info(f"Date: {claim_date}")
        
        # Get model prediction
        try:
            prediction = ask_model(
                model_name,
                claim_text,
                claim_date=claim_date,
                logger=logger,
                max_retries=MAX_RETRIES,
                initial_delay=INITIAL_RETRY_DELAY
            )
            
            result_entry = claim_entry.copy()
            
            if prediction:
                result_entry["prediction"] = prediction
                result_entry["prediction_error"] = None
                successful += 1
                logger.info(f"Prediction: {prediction}")
            else:
                result_entry["prediction"] = None
                result_entry["prediction_error"] = "Failed to get valid prediction"
                failed += 1
                logger.warning("Failed to get valid prediction")
            
            results.append(result_entry)
            
        except Exception as e:
            logger.error(f"Exception processing claim {idx}: {e}", exc_info=True)
            result_entry = claim_entry.copy()
            result_entry["prediction"] = None
            result_entry["prediction_error"] = str(e)
            results.append(result_entry)
            failed += 1
        
        # Update progress bar
        pbar.set_postfix({
            'successful': successful,
            'failed': failed,
            'success_rate': f"{(successful / (idx + 1) * 100):.1f}%"
        })
        
        # Sleep between calls
        if sleep_between > 0:
            time.sleep(sleep_between)
    
    pbar.close()
    
    t1 = time.time()
    elapsed = t1 - t0
    
    # Save results
    logger.info(f"\nSaving results to {output_file}...")
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    
    # Compute statistics
    stats = {
        "input_file": str(input_file),
        "output_file": str(output_file),
        "total_claims": total_claims,
        "successful_predictions": successful,
        "failed_predictions": failed,
        "success_rate": successful / total_claims if total_claims > 0 else 0.0,
        "runtime_seconds": elapsed,
        "claims_per_second": total_claims / elapsed if elapsed > 0 else 0.0,
        "model": model_name,
    }
    
    # Log statistics
    logger.info(f"\n{'='*80}")
    logger.info("PROCESSING COMPLETE")
    logger.info(f"{'='*80}")
    logger.info(f"Total claims: {total_claims}")
    logger.info(f"Successful predictions: {successful}")
    logger.info(f"Failed predictions: {failed}")
    logger.info(f"Success rate: {stats['success_rate']:.2%}")
    logger.info(f"Runtime: {elapsed:.2f} seconds")
    logger.info(f"Processing speed: {stats['claims_per_second']:.2f} claims/sec")
    logger.info(f"{'='*80}")
    
    return stats

print("✓ Inference pipeline defined successfully.")

✓ Inference pipeline defined successfully.


<a id="run"></a>
# 5. Run Inference

Execute inference on both FEVER dataset files.

In [5]:
# Setup main logger
main_log_file = OUTPUT_DIR / "main_run_log.txt"
main_logger = setup_logger(main_log_file)

print("="*80)
print("FEVER DATASET INFERENCE")
print("="*80)
print(f"Model: {MODEL_NAME}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Log file: {main_log_file}")
print("="*80)

all_stats = []

for input_file in INPUT_FILES:
    if not input_file.exists():
        print(f"⚠️  File not found: {input_file}")
        main_logger.warning(f"File not found: {input_file}")
        continue
    
    # Generate output filename
    output_file = OUTPUT_DIR / f"{input_file.stem}_with_predictions.json"
    
    print(f"\n{'='*80}")
    print(f"Processing: {input_file.name}")
    print(f"{'='*80}")
    
    # Process file
    stats = process_fever_file(
        input_file=input_file,
        output_file=output_file,
        model_name=MODEL_NAME,
        logger=main_logger,
        sleep_between=SLEEP_BETWEEN_CALLS
    )
    
    all_stats.append(stats)
    
    # Print summary
    print(f"\n✓ Completed: {input_file.name}")
    print(f"  - Total claims: {stats['total_claims']}")
    print(f"  - Successful: {stats['successful_predictions']}")
    print(f"  - Failed: {stats['failed_predictions']}")
    print(f"  - Success rate: {stats['success_rate']:.2%}")
    print(f"  - Output saved to: {output_file}")

# Save overall statistics
summary_file = OUTPUT_DIR / "inference_summary.json"
with open(summary_file, 'w', encoding='utf-8') as f:
    json.dump({
        "model": MODEL_NAME,
        "files_processed": len(all_stats),
        "total_claims": sum(s['total_claims'] for s in all_stats),
        "total_successful": sum(s['successful_predictions'] for s in all_stats),
        "total_failed": sum(s['failed_predictions'] for s in all_stats),
        "overall_success_rate": sum(s['successful_predictions'] for s in all_stats) / sum(s['total_claims'] for s in all_stats) if all_stats else 0.0,
        "file_statistics": all_stats
    }, f, indent=2)

print(f"\n{'='*80}")
print("ALL FILES PROCESSED")
print(f"{'='*80}")
print(f"Summary saved to: {summary_file}")
print(f"Log saved to: {main_log_file}")
print("="*80)

FEVER DATASET INFERENCE
Model: meta.llama3-1-70b-instruct-v1:0
Output directory: results_fever
Log file: results_fever\main_run_log.txt

Processing: test_2023_2024.json


Processing test_2023_2024.json:   0%|          | 0/2215 [00:00<?, ?claim/s]

KeyboardInterrupt: 

<a id="results"></a>
# 6. Results Analysis

Analyze the inference results and display statistics.

In [ ]:
# Load and display summary
summary_file = OUTPUT_DIR / "inference_summary.json"

if summary_file.exists():
    with open(summary_file, 'r', encoding='utf-8') as f:
        summary = json.load(f)
    
    print("="*80)
    print("INFERENCE SUMMARY")
    print("="*80)
    print(f"Model: {summary['model']}")
    print(f"Files processed: {summary['files_processed']}")
    print(f"Total claims: {summary['total_claims']}")
    print(f"Successful predictions: {summary['total_successful']}")
    print(f"Failed predictions: {summary['total_failed']}")
    print(f"Overall success rate: {summary['overall_success_rate']:.2%}")
    print("="*80)
    
    print("\nPer-file statistics:")
    print("-"*80)
    for stat in summary['file_statistics']:
        print(f"\n{Path(stat['input_file']).name}:")
        print(f"  Total: {stat['total_claims']}")
        print(f"  Successful: {stat['successful_predictions']}")
        print(f"  Failed: {stat['failed_predictions']}")
        print(f"  Success rate: {stat['success_rate']:.2%}")
        print(f"  Runtime: {stat['runtime_seconds']:.2f}s")
        print(f"  Speed: {stat['claims_per_second']:.2f} claims/sec")
else:
    print("⚠️  Summary file not found. Please run the inference first.")

In [ ]:
# Analyze prediction distribution
from collections import Counter

print("\n" + "="*80)
print("PREDICTION DISTRIBUTION")
print("="*80)

for input_file in INPUT_FILES:
    output_file = OUTPUT_DIR / f"{input_file.stem}_with_predictions.json"
    
    if not output_file.exists():
        continue
    
    with open(output_file, 'r', encoding='utf-8') as f:
        results = json.load(f)
    
    predictions = [r.get('prediction') for r in results if r.get('prediction') is not None]
    pred_counts = Counter(predictions)
    
    print(f"\n{input_file.name}:")
    print("-"*80)
    for label in CANON_LABELS:
        count = pred_counts.get(label, 0)
        percentage = (count / len(predictions) * 100) if predictions else 0
        print(f"  {label}: {count} ({percentage:.1f}%)")
    print(f"  Total valid predictions: {len(predictions)}")

---

## Summary

This notebook processes FEVER dataset files and adds model predictions to each claim.

**Key Features:**
- Loads claims from AVeriTeC_FEVER JSON files
- Uses Llama 3.1 70B Instruct model for classification
- Handles API retries and error cases
- Saves results with predictions in the same JSON format
- Provides detailed logging and statistics

**Output Files:**
- `results_fever/test_2023_2024_with_predictions.json` - 2023-2024 claims with predictions
- `results_fever/test_2025_with_predictions.json` - 2025 claims with predictions
- `results_fever/inference_summary.json` - Overall statistics
- `results_fever/main_run_log.txt` - Detailed execution log

---